<a href="https://colab.research.google.com/github/CesarChaMal/ollama/blob/main/ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Download and run the Ollama Linux install script
!curl https://ollama.ai/install.sh | sh
!command -v systemctl >/dev/null && sudo systemctl stop ollama

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8354    0  8354    0     0   9638      0 --:--:-- --:--:-- --:--:--  9635>>> Downloading ollama...
100  8354    0  8354    0     0   8981      0 --:--:-- --:--:-- --:--:--  8973
############################################################################################# 100.0%
>>> Installing ollama to /usr/local/bin...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 0.0.0.0:11434.
>>> Install complete. Run "ollama" from the command line.
System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down


In [ ]:
!pip install aiohttp pyngrok

import os
import asyncio
from aiohttp import ClientSession

# Set LD_LIBRARY_PATH so the system NVIDIA library becomes preferred
# over the built-in library. This is particularly important for
# Google Colab which installs older drivers
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

from pyngrok import ngrok
# Set your authtoken (replace 'YOUR_AUTHTOKEN' with the token you got from ngrok)
ngrok.set_auth_token("YOUR_AUTHTOKEN")

# Open a HTTP tunnel on the specified port
public_url = ngrok.connect(8443, "http")
print("Public URL:", public_url)

async def run(cmd):
  '''
  run is a helper function to run subcommands asynchronously.
  '''
  print('>>> starting', *cmd)
  p = await asyncio.subprocess.create_subprocess_exec(
      *cmd,
      stdout=asyncio.subprocess.PIPE,
      stderr=asyncio.subprocess.PIPE,
  )

  async def pipe(lines):
    async for line in lines:
      print(line.strip().decode('utf-8'))

  await asyncio.gather(
      pipe(p.stdout),
      pipe(p.stderr),
  )


await asyncio.gather(
    run(['ollama', 'serve']),
    run(['ngrok', 'http', '--log', 'stderr', '11434']),
)

>>> starting ollama serve
>>> starting ngrok http --log stderr 11434
2024/01/07 13:00:46 images.go:834: total blobs: 0
2024/01/07 13:00:46 images.go:841: total unused blobs removed: 0
2024/01/07 13:00:46 routes.go:929: Listening on 127.0.0.1:11434 (version 0.1.18)
t=2024-01-07T13:00:46+0000 lvl=info msg="no configuration paths supplied"
t=2024-01-07T13:00:46+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
t=2024-01-07T13:00:46+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
t=2024-01-07T13:00:46+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
t=2024-01-07T13:00:47+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session obj=csess id=cf3aff0d337e err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your

The previous cell starts two processes, `ollama` and `ngrok`. The log output will show a line like the following which describes the external address.

```
t=2023-11-12T22:55:56+0000 lvl=info msg="started tunnel" obj=tunnels name=command_line addr=http://localhost:11434 url=https://8249-34-125-179-11.ngrok.io
```

The external address in this case is `https://8249-34-125-179-11.ngrok.io` which can be passed into `OLLAMA_HOST` to access this instance.

```bash
export OLLAMA_HOST=https://8249-34-125-179-11.ngrok.io
ollama list
ollama run mistral
```